# 第15课 · 从结果倒推原因——高斯消元（Gaussian elimination）与 Ax=b 的三种命运

**学习目标**
1. 把线性方程组写成增广矩阵（augmented matrix） `[A|b]`，掌握矩阵方程 `Ax = b` 的矩阵形式
2. 掌握三种初等行变换（交换 ER1、倍乘 ER2、消元 ER3）并在具体例题中手动执行
3. 用 `rank(A)` 和 `rank([A|b])` 分类方程组的解：唯一解、无穷多解、无解
4. 实现 `classify_system(A, b)` 函数，返回 `'unique'`/`'many'`/`'none'`，通过三个补充例题验证
5. 理解 Aurora LPC 系数估计的超定系统（overdetermined system）本质，`np.linalg.lstsq` 求最小二乘解

**为什么对 Aurora 重要**：LPC 把声音建模为线性系统（linear system） `Ax = b`，其中 `x` 是滤波器系数；系数估计用最小二乘法（least squares method，LS） `x = (AᵀA)⁻¹Aᵀb`，能否得到唯一解直接取决于 `rank(A)`。

← **上一课**　[L14 · 特征值与 SVD](L14_eigen_svd.ipynb)

> 上节课学习了 **特征值与 SVD**：找到矩阵偏爱的方向，用 SVD 打开低秩世界。  
> 本课将探讨 **高斯消元**。

## 本课剧情：已知目的地，倒推出你走了哪条路

你知道某个线性变换的结果（`b`），也知道变换的规则（矩阵 `A`）。  
问题：原始输入 `x` 是什么？这就是 `Ax = b`。

现实中随处可见：
- 语音增强：已知麦克风录音 b，房间混响矩阵 A，还原干净语音 x
- 线性回归：已知数据 b，特征矩阵 A，找权重向量 x = argmin‖Ax-b‖
- 电路分析：已知电压 b，阻抗矩阵 A，求电流 x

方程组可能有三种命运：
- **唯一解**：A 满秩，`rank(A) = rank([A|b]) = n`
- **无数解**：A 列线性相关，消元后有自由变量
- **无解**：b 不在 A 的列空间中，方程矛盾

本课你将实现 `classify_system(A, b)` 来自动判断哪种情况。

## 1. Ax=b：用列的线性组合拼出 b

看 `A @ x = b` 的"视角翻转"：

**列视角**：`A @ x = x[0]·col₀ + x[1]·col₁ + ... + x[n-1]·colₙ₋₁`  
→ 求解 `Ax=b` = 找一组系数 `x`，让 A 的各列加权组合精确等于 `b`

**例子**：A = [[2, 1], [0, 1]]，b = [5, 3]  
问：能用 x[0]×[2,0] + x[1]×[1,1] 拼出 [5,3] 吗？  
答案：x[0]=1, x[1]=3 → 1×[2,0] + 3×[1,1] = [2,0]+[3,3] = [5,3] ✓

**高斯消元的作用**：通过初等行变换，把增广矩阵 [A|b] 化成行阶梯形（REF），从最后一行"倒代入"（back substitution）逐个求解。

**rank 判断可解性**：
- `rank(A) = rank([A|b]) = n` → 唯一解
- `rank(A) = rank([A|b]) < n` → 无数解
- `rank(A) < rank([A|b])` → 无解

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
x = np.array([3.0, 2.0])
b = A @ x

print('A 的第 1 列:', A[:, 0])
print('A 的第 2 列:', A[:, 1])
print('x =', x)
print('b = 3 * col1 + 2 * col2 =', b)


## 1.1 再进入一个三元例题

下面的三元方程组只是同一件事的更大版本：`A` 有三列，`x` 有三个系数，目标是拼出右侧的 `b`。


$$2x + y - 2z = 10,\quad 3x + 2y + 2z = 1,\quad 5x + 4y + 3z = 4$$

课件答案：唯一解 $x=1,\,y=2,\,z=-3$。


## 符号入口：先看形状，再看运算

`A` 是 `(m, n)` 矩阵，`x` 是 `(n,)` 向量（vector），`A @ x` 得到 `(m,)` 向量。`m` 是方程个数，`n` 是未知数个数，这两个数的大小关系决定了方程组是超定（overdetermined）、恰定（square）还是欠定（underdetermined）。

In [ ]:
import numpy as np
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
print('解 =', np.linalg.solve(A, b), ' (课件 [1, 2, -3])')

## 动手观察：先看 shape，再看数值

运行这段代码，注意 `A.shape`、`x.shape`、`b.shape` 三者的关系——`m`、`n` 的具体值决定了线性系统有多少方程、多少未知数，也决定了能否用 `np.linalg.solve` 求解。

In [ ]:
import numpy as np

# 利用 numpy 对不同秩矩阵观察秩的变化（解的分类留给习题 4 实现）
cases = [
    ('唯一解', np.array([[2.,1.],[0.,3.]]), np.array([5.,6.])),
    ('无穷多解', np.array([[1.,2.],[2.,4.]]), np.array([3.,6.])),
    ('无解', np.array([[1.,1.],[1.,1.]]), np.array([2.,3.])),
]
for label, A, b in cases:
    rank_A  = np.linalg.matrix_rank(A)
    rank_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))
    n = A.shape[1]
    print(f'{label}: rank(A)={rank_A}, rank([A|b])={rank_Ab}, n={n}')
# 观察：三行数字对应三种情形，习题 4 就是把这些比较关系编码进 classify_system。


## 代码实验：同一个 A，不同的 x，看输出如何线性变化

用几个不同的 `x` 向量计算 `A @ x`，确认矩阵乘法是线性映射——这是高斯消元能逐步消去未知数的前提。

In [ ]:
import numpy as np

# 消元后验证：Ax=b 的残差
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10.,1.,4.])
x = np.linalg.solve(A, b)
residual = np.linalg.norm(A @ x - b)
print(f'解 x = {np.round(x,6)}')
print(f'残差 ||Ax-b|| = {residual:.2e}  （接近0=正确）')
print(f'验证：A@x = {np.round(A@x,6)}')


## 1.2 `np.linalg.solve` 给出答案，高斯消元解释过程

先用工具确认答案，再看增广矩阵如何一步步变形。这样后面的行变换有一个明确目标：把未知数逐个分离出来。


## 2. 初等行变换 ER1 / ER2 / ER3 与增广矩阵

- **ER1** 交换两行  `Ri ↔ Rj`
- **ER2** 某行乘非零常数  `Ri → k·Ri`
- **ER3** 某行加上另一行的 k 倍  `Ri → Ri + k·Rj`

下面手动做课件那一步：`R2→2R2−3R1`, `R3→2R3−5R1`，把第一列下方消成 0。


In [ ]:
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
aug = np.column_stack([A, b])      # 增广矩阵 [A | b]
print('增广矩阵:\n', aug)
aug[1] = 2*aug[1] - 3*aug[0]       # ER3 + ER2
aug[2] = 2*aug[2] - 5*aug[0]
print('\n消元后(对照补充例题):\n', aug)


## 3. 三种情形：用秩(rank)判断

比较系数矩阵 A 的秩与增广矩阵 [A|b] 的秩、未知数个数 n：
- `rank(A) = rank([A|b]) = n` → **唯一解**
- `rank(A) = rank([A|b]) < n` → **无穷多解**
- `rank(A) < rank([A|b])` → **无解**（矛盾）


## 4. ✏️ 实现 `classify_system(A, b)`

**推理路线**：
1. 把 A 和 b 拼成增广矩阵 [A|b]，分别计算 `rA = rank(A)` 和 `rAb = rank([A|b])`。
2. 若 `rA < rAb`：b 不在 A 的列空间里，方程矛盾，返回 `'none'`。
3. 若 `rA == rAb == n`：每个未知数对应一个独立约束，返回 `'unique'`。
4. 若 `rA == rAb < n`：约束数不够，存在自由变量，返回 `'many'`。

**参考输入输出**：
- `A=[[2,1],[1,3]], b=[5,10]` → `'unique'`（rank=2=n）
- `A=[[1,2],[2,4]], b=[3,6]` → `'many'`（rank=1<n=2，b 在列空间里）
- `A=[[1,2],[2,4]], b=[3,7]` → `'none'`（rank(A)=1 < rank([A|b])=2）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


写 `classify_system` 前明确三件事：
- 输入：`A`（系数矩阵，`(m, n)`）、`b`（右端向量，`(m,)`）
- 关键步骤：计算 `rA = rank(A)`，`rAb = rank([A|b])`，再与 `n = A.shape[1]` 比较
- 返回：字符串 `'unique'`、`'many'` 或 `'none'`

In [ ]:
def classify_system(A, b):
    A = np.asarray(A, float); b = np.asarray(b, float)
    # ✏️ TODO: 用秩判断解的情形，返回 'unique'/'many'/'none'
    raise NotImplementedError(
        "TODO: implement classify_system → return 'unique'/'many'/'none'"
    )


In [ ]:
# 三个补充例题 (3×3) + 超定例题 (4×2，对应 Aurora LPC 超定场景)
U  = (np.array([[2,1,-2],[3,2,2],[5,4,3]],float), np.array([10,1,4],float))
M  = (np.array([[1,2,-3],[2,-1,4],[4,3,-2]],float), np.array([6,2,14],float))
N  = (np.array([[1,2,-3],[3,-1,2],[5,3,-4]],float), np.array([-1,7,2],float))
# 超定 (4 方程, 2 未知数)：rank=2=n，b 在列空间内 → 'unique'
OD = (np.array([[1.,0.],[0.,1.],[1.,1.],[2.,1.]]), np.array([1.,1.,2.,3.]))

try:
    print('例1 (unique):', classify_system(*U),  '  预期: unique')
    print('例2 (many):  ', classify_system(*M),  '  预期: many')
    print('例3 (none):  ', classify_system(*N),  '  预期: none')
    print('超定 (unique):', classify_system(*OD), '  预期: unique (最小二乘意义)')
    assert classify_system(*U)  == 'unique', '例1 失败'
    assert classify_system(*M)  == 'many',   '例2 失败'
    assert classify_system(*N)  == 'none',   '例3 失败'
    assert classify_system(*OD) == 'unique', '超定例题失败'
    print('\n✅ 通过：三个补充例题 + 超定例题的解类型检查完成。')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  任务未完成：{e}')
    print('请先实现上方 classify_system 函数再运行此单元格。')


### 参考对照：`classify_system` 核心逻辑（完成习题后再展开）

> 以下是可运行的参考实现，供完成习题后核对。未完成前请勿提前查看。


In [ ]:
# ── 参考实现（仅供完成习题后对照）──
def _classify_system_ref(A, b):
    A  = np.asarray(A, float); b = np.asarray(b, float)
    rA  = np.linalg.matrix_rank(A)
    rAb = np.linalg.matrix_rank(np.column_stack([A, b]))
    n   = A.shape[1]
    if rA < rAb:
        return 'none'
    elif rA == rAb == n:
        return 'unique'
    else:
        return 'many'

# 验证参考实现本身正确
assert _classify_system_ref(*U)  == 'unique'
assert _classify_system_ref(*M)  == 'many'
assert _classify_system_ref(*N)  == 'none'
assert _classify_system_ref(*OD) == 'unique'
print('参考实现自检通过 ✅')


**🔗 Aurora 连接**：线性回归的正规方程 `(XᵀX)w = Xᵀy` 就是一个线性系统；可解性对应模型何时有唯一最优解。

**补充例题对应**：增广矩阵、阶梯形、初等行变换、解的分类。


## 🎨 图示：Ax=b 看成 A 各列的线性组合 (补充例题 1)


In [ ]:
from aurora.laviz import style, mat_times_vec
style()
mat_times_vec([[2,1,-2],[3,2,2],[5,4,3]],[1,2,-3]);  # = b=[10,1,4]

In [ ]:
import numpy as np

# 参数实验：A 的行向量越接近线性相关，条件数爆炸
for eps in [0.1, 0.01, 0.001, 1e-6]:
    A = np.array([[1., 1.],[1., 1.+eps]])
    kappa = np.linalg.cond(A)
    print(f'eps={eps:.0e}  κ(A)={kappa:.2e}')
print('→ eps→0 时行向量几乎平行，Ax=b 数值求解越来越不稳定。')


## 参数实验：奇异矩阵如何触发错误，lstsq 如何补救

把 A 的行列式设为 0（如 `A=[[1,2],[2,4]]`），调用 `np.linalg.solve(A, b)` 会触发 `LinAlgError: Singular matrix`——奇异矩阵没有逆，无法得到唯一解。再用 `np.linalg.lstsq(A, b, rcond=None)` 处理同一个矩阵，它不报错，返回最小范数解（有无穷解时）或残差最小的近似解（方程矛盾时）。每次只改一个量，先预测 `solve` 还是 `lstsq` 会成功，再运行验证。

In [ ]:
import numpy as np

A_sing = np.array([[1., 2.],
                   [2., 4.]])   # 行列式 = 0，奇异矩阵

# ── 场景 1：一致方程（b 在列空间内） ──
b_consistent   = np.array([3., 6.])  # [3,6] = 3*[1,2]
b_inconsistent = np.array([3., 7.])  # [3,7] 不在 [1,2] 张成的空间里

print('=== np.linalg.solve（奇异矩阵应抛出 LinAlgError）===')
for label, bv in [('一致', b_consistent), ('矛盾', b_inconsistent)]:
    try:
        x = np.linalg.solve(A_sing, bv)
        print(f'  {label} b: 意外成功，x={x}')
    except np.linalg.LinAlgError as e:
        print(f'  {label} b: ✅ 捕获 LinAlgError — {e}')

print()
print('=== np.linalg.lstsq（不报错，返回最小范数/最小残差解）===')
for label, bv in [('一致', b_consistent), ('矛盾', b_inconsistent)]:
    x_ls, res, rank, sv = np.linalg.lstsq(A_sing, bv, rcond=None)
    residual = np.linalg.norm(A_sing @ x_ls - bv)
    print(f'  {label} b: x_ls={np.round(x_ls,4)},  残差={residual:.4f},  rank={rank}')
print()
print('结论：solve 在奇异矩阵时失败；lstsq 始终给出答案，')
print('      但一致方程残差≈0，矛盾方程残差>0——用 classify_system 可提前判断。')


## 本课收束

现在可以用 `np.linalg.solve(A, b)` 求唯一解，用 `classify_system(A, b)` 判断方程组的可解性。Aurora 的 LPC 系数估计把录音片段的自相关矩阵当作 `A`，解这个线性系统得到预测滤波器 `x`；`rank(A)` 不足时需要切换到最小二乘形式。下一节介绍行列式与逆矩阵，给出 `rank` 降低时系统退化的几何解释。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：线性方程组手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：方程组 `2x + y = 5, x + 3y = 8`  
写出增广矩阵 `[A|b]`，消元后求 x 和 y。

**问 2**：A = [[1, 2], [2, 4]]，b = [1, 3]，  
这个方程组有唯一解、无数解还是无解？（手算 rank(A) 和 rank([A|b])）

**问 3**：A = [[1, 2], [2, 4]]，b = [2, 4]，  
和问 2 的矩阵完全一样，但 b 变了——现在是哪种情况？

**问 4**：线性方程组 `Ax=b` 中，若 A 是 4×4 矩阵且 `rank(A) = 4`，  
则对任意 b，方程组有几个解？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：2x+y=5, x+3y=8
A1 = np.array([[2., 1.], [1., 3.]])
b1 = np.array([5., 8.])
x1 = np.linalg.solve(A1, b1)
assert np.allclose(A1 @ x1, b1, atol=1e-10)
print(f"Q1 ✅  2x+y=5, x+3y=8 → x={x1[0]:.4f}, y={x1[1]:.4f}")
print(f"       验证 A@x = {np.round(A1 @ x1, 4)} = b = {b1}")

# 问2：rank 判断（A行线性相关，b矛盾）
A2 = np.array([[1., 2.], [2., 4.]])
b2_no = np.array([1., 3.])
rA2 = np.linalg.matrix_rank(A2)
rAb2 = np.linalg.matrix_rank(np.column_stack([A2, b2_no]))
print(f"\nQ2 ✅  rank(A)={rA2}, rank([A|b])={rAb2} → 无解（rank 不等，b 不在列空间）")
try:
    cs = classify_system(A2, b2_no)
    assert 'no' in cs.lower() or '无' in cs or 'inconsistent' in cs.lower()
    print(f"       classify_system → '{cs}'")
except (NotImplementedError, TypeError):
    print(f"       (classify_system 待实现)")

# 问3：同A，b变为[2,4]（在列空间）
b2_inf = np.array([2., 4.])
rAb3 = np.linalg.matrix_rank(np.column_stack([A2, b2_inf]))
print(f"\nQ3 ✅  rank(A)={rA2}, rank([A|b])={rAb3}, n=2 → rank(A)<n → 无数解")
try:
    cs3 = classify_system(A2, b2_inf)
    assert 'inf' in cs3.lower() or '无数' in cs3 or 'many' in cs3.lower()
    print(f"       classify_system → '{cs3}'")
except (NotImplementedError, TypeError):
    print(f"       (classify_system 待实现)")

# 问4：4×4满秩 → 唯一解
A4 = np.eye(4) * 2 + np.random.default_rng(0).standard_normal((4,4)) * 0.1
rank_A4 = np.linalg.matrix_rank(A4)
assert rank_A4 == 4, f"矩阵秩应为4"
print(f"\nQ4 ✅  4×4 满秩矩阵（rank={rank_A4}=n=4）→ 唯一解（对任意 b）")
print("\n🎉 线性方程组白板挑战通过！rank 判断可解性已内化。")

In [ ]:
# ✏️ 本课自评
l15_review = {
    "classify_system_implemented": None,  # classify_system 实现并通过断言？True/False
    "gaussian_elimination_understood": None,  # 理解增广矩阵消元过程？True/False
    "rank_solvability_rule":       None,  # 能用 rank 判断唯一/无数/无解？True/False
    "column_space_intuition":      None,  # 理解"b 在 A 的列空间"的含义？True/False
    "whiteboard_passed":           None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l15_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l15_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L15 全部通关！进入 L16：行列式与逆矩阵')

---

→ **下一课**　[L16 · 行列式与逆矩阵](L16_determinant_inverse.ipynb)

> 下节课将学习 **行列式与逆矩阵**：det(A) 的几何含义（面积缩放），求逆与何时不可逆。